# 04 — Merge bảng đã xử lí vào processed/

Ghép dọc các bảng khách–tháng do notebook 03 xuất. Mỗi bảng đã nối feature trx, geo,
dialog và nhãn bằng khóa `(client_id, cutoff)`. Không join trực tiếp log sự kiện hoặc
gộp currency/category sidecar nhiều dòng vào một dòng khách–tháng.

**Notebook nguồn chưa thực thi**, output trống. Chạy từ đầu sau khi notebook 03 có
manifest `ready` và schema v2. Mặc định 12 tháng 02/2022–01/2023; không bỏ khách
cold-start, dòng thiếu nhãn, không cap/impute hoặc fit mô hình.

| Bước | Mục tiêu | Đầu ra |
| --- | --- | --- |
| 1–2 | Môi trường và đường dẫn | Python >=3.11; nguồn output notebook 03 |
| 3 | Định nghĩa đầy đủ helper dùng chung | Schema v2, kiểm tra dữ liệu, hàm merge |
| 4 | Kiểm kê nguồn | Số dòng và nhãn dự kiến theo tháng |
| 5 | Kiểm tra, ghép theo batch, đối soát | `processed/propensity.parquet` |
| 6 | Xác nhận trạng thái | Manifest ready, schema và protocol |

Cài dependency trên máy chạy: `python -m pip install -e ".[prepare,notebook]"`.
Thư mục `/processed/` được Git ignore toàn bộ, gồm cả Parquet và metadata.


## 1. Môi trường


In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if sys.version_info < (3, 11):
    raise RuntimeError("Cần Python >=3.11.")
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục repo hoặc notebooks/.")
print(PROJECT_ROOT)


## 2. Đường dẫn

Chọn output mới nếu chạy lại; không ghi đè hoặc xóa kết quả trước.


In [ ]:
SOURCE_DIR = PROJECT_ROOT / "outputs" / "propensity_processing_v3"
OUTPUT_DIR = PROJECT_ROOT / "processed"
BATCH_SIZE = 10_000
# Có thể đổi OUTPUT_DIR = PROJECT_ROOT / "processed" / "run_02" khi cần một lần chạy mới.
# Đường dẫn nguồn có thể trỏ tới bộ đã chuẩn bị trên máy khác, cần giữ work.duckdb.
print("Source:", SOURCE_DIR)
print("Merged output:", OUTPUT_DIR)


## 3. Helper đầy đủ

Các cell định nghĩa được đồng bộ với mã nguồn; không đọc dữ liệu khi định nghĩa.


In [ ]:
"""Validated configuration; all relative paths are relative to the JSON file."""
from dataclasses import asdict, dataclass
from datetime import date
import json
from pathlib import Path
import re


def month_start(value: str) -> date:
    if not isinstance(value, str) or not re.fullmatch(r"\d{4}-\d{2}", value):
        raise ValueError("Months must use YYYY-MM")
    return date.fromisoformat(value + "-01")


@dataclass(frozen=True)
class PreparationConfig:
    data_root: Path
    output_dir: Path
    first_month: str = "2022-02"
    last_month: str = "2023-01"
    windows_days: tuple[int, ...] = (30, 84)
    memory_limit: str = "6GB"
    threads: int = 2
    label_delay_days: int = 0
    calibration_month: str = "2022-12"
    test_month: str = "2023-01"
    validation_months: tuple[str, ...] = ("2022-09", "2022-10", "2022-11")
    include_dialog_embeddings: bool = False
    embedding_size: int = 768
    embedding_window_days: int = 84

    def __post_init__(self):
        for key in ("data_root", "output_dir"):
            object.__setattr__(self, key, Path(getattr(self, key)).resolve())
        for key in ("windows_days", "validation_months"):
            object.__setattr__(self, key, tuple(getattr(self, key)))
        if (self.output_dir == self.data_root or self.data_root in self.output_dir.parents
                or self.output_dir in self.data_root.parents):
            raise ValueError("Output and raw-data trees must not overlap")
        for key in ("threads", "embedding_size", "embedding_window_days"):
            value = getattr(self, key)
            if type(value) is not int or value < 1:
                raise ValueError(f"{key} must be a positive integer")
        if type(self.label_delay_days) is not int or self.label_delay_days < 0:
            raise ValueError("label_delay_days must be a nonnegative integer")
        if type(self.include_dialog_embeddings) is not bool:
            raise ValueError("include_dialog_embeddings must be boolean")
        if not self.windows_days or any(type(w) is not int or w < 1 for w in self.windows_days):
            raise ValueError("windows_days must contain positive integers")
        if len(set(self.windows_days)) != len(self.windows_days):
            raise ValueError("Duplicate windows")
        if self.include_dialog_embeddings and self.embedding_window_days not in self.windows_days:
            raise ValueError("embedding_window_days must occur in windows_days")
        if not isinstance(self.memory_limit, str) or not re.fullmatch(r"[1-9]\d*(MB|GB)", self.memory_limit):
            raise ValueError("memory_limit must be a positive integer followed by MB or GB")
        months = [self.first_month, self.last_month, self.calibration_month, self.test_month,
                  *self.validation_months]
        for value in months:
            month_start(value)
        if not self.first_month <= self.calibration_month < self.test_month <= self.last_month:
            raise ValueError("Require first <= calibration < test <= last")
        if (not self.validation_months or tuple(sorted(set(self.validation_months))) != self.validation_months
                or not all(self.first_month < m < self.calibration_month for m in self.validation_months)):
            raise ValueError("Validation months must be ordered, unique, and before calibration")

    def to_dict(self):
        result = asdict(self)
        for key in ("data_root", "output_dir"):
            result[key] = str(result[key])
        return result


def load_config(path: str | Path) -> PreparationConfig:
    path = Path(path).resolve()
    options = json.loads(path.read_text(encoding="utf-8-sig"))
    for key in ("data_root", "output_dir"):
        options[key] = (path.parent / options[key]).resolve()
    return PreparationConfig(**options)


In [ ]:
"""Source layout and anonymized MBD-mini columns."""
SOURCES = {"trx": "detail/trx", "geo": "detail/geo", "dialog": "detail/dialog"}
LABELS = tuple(f"target_{i}" for i in range(1, 5))
CATEGORIES = ("event_type", "event_subtype", "src_type11", "src_type12", "dst_type11",
              "dst_type12", "src_type21", "src_type22", "src_type31", "src_type32")


def feature_schema(config, *, window_tags=None) -> dict:
    """Approved names, order and DuckDB types; never infer predictors from input."""
    excluded = ["client_id", "cutoff", "y_any", *LABELS, "supervised_eligible"]
    dtypes = {"client_id": "VARCHAR", "cutoff": "TIMESTAMP", "y_any": "TINYINT",
              **{c: "DOUBLE" for c in LABELS}, "supervised_eligible": "BOOLEAN"}
    for source in SOURCES:
        tags = window_tags if window_tags is not None else [*(str(d) + 'd' for d in config.windows_days), 'all']
        for tag in tags:
            prefix = f"{source}_{tag}_"
            window = {
                "source_window_overlap_days": "DOUBLE", "history_truncated": "TINYINT",
                "has_event": "TINYINT", "event_count": "BIGINT", "active_days": "BIGINT",
                "days_since_last_event": "DOUBLE", "days_since_first_recorded_event": "DOUBLE",
            }
            if source == "trx":
                window.update({name: "BIGINT" for name in (
                    "main_count", "foreign_count", "known_currency_count", "currency_missing_count",
                    "currency_invalid_count", "amount_missing_count", "amount_invalid_count",
                    "main_amount_count", "main_negative_amount_count")})
                window.update({name: "DOUBLE" for name in (
                    "main_amount_sum", "main_amount_mean", "main_amount_median", "main_amount_max",
                    "main_amount_abs_sum", "main_amount_signed_log_mean", "foreign_fraction",
                    "foreign_fraction_known_currency")})
                for cat in CATEGORIES:
                    window.update({cat + "_nunique": "BIGINT", cat + "_missing_count": "BIGINT"})
            elif source == "geo":
                for i in (4, 5, 6):
                    window.update({f"geohash_{i}_nunique": "BIGINT", f"geohash_{i}_missing_count": "BIGINT"})
            window["log1p_event_count"] = "DOUBLE"
            if source == "trx":
                window.update({"log1p_main_amount_abs_sum": "DOUBLE", "has_foreign": "TINYINT",
                               "event_type_top_share": "DOUBLE", "event_subtype_top_share": "DOUBLE"})
            elif source == "geo":
                window["top_geohash_6_share"] = "DOUBLE"
            elif config.include_dialog_embeddings and tag == str(config.embedding_window_days) + 'd':
                window["valid_embedding_count"] = "BIGINT"
                for pooling in ("mean", "last"):
                    window.update({f"embedding_{pooling}_{i}": "DOUBLE"
                                   for i in range(1, config.embedding_size + 1)})
            dtypes.update({prefix + name: dtype for name, dtype in window.items()})
        dtypes[source + "_ever_seen_source"] = "TINYINT"
        if source == "dialog":
            dtypes["dialog_has_dialog"] = "TINYINT"
    dtypes["cold_start"] = "TINYINT"
    predictors = [c for c in dtypes if c not in excluded]
    return {
        "schema_version": 2 if window_tags is None else 3, "columns": list(dtypes), "dtypes": dtypes,
        "keys": ["client_id", "cutoff"], "target": "y_any", "excluded_from_predictors": excluded,
        "predictors": predictors,
        "groups": {s: [c for c in predictors if c.startswith(s + '_')] for s in SOURCES},
        "ablation_groups": {
            "trx": [c for c in predictors if c.startswith('trx_')],
            "source_metadata": [c for c in predictors if c.startswith(('geo_', 'dialog_'))
                                and 'geohash_' not in c and 'embedding_' not in c],
            "geo_content": [c for c in predictors if c.startswith('geo_') and 'geohash_' in c],
            "dialog_content": [c for c in predictors if c.startswith('dialog_') and 'embedding_' in c],
            "cohort_flags": ["cold_start"],
        },
        "missing_numeric": "Keep NaN; counts are zero only within calendar overlap with the source",
        "currency_sidecar": "Separate units; do not sum amounts across currency values",
        "feature_definitions": {
            "*_has_foreign": "1 if known foreign observed; 0 if no logs in covered window or known main and no known foreign; null if only unknown currency or no source overlap",
            "*_foreign_fraction": "Observed foreign logs / all logs; 0 does not prove all currencies were main",
            "*_foreign_fraction_known_currency": "Known foreign logs / logs with valid currency; null when denominator is zero",
        },
        "scope_note": "Calendar extract overlap is not continuous observation of each customer",
    }


def validate_feature_columns(description, schema):
    """Reject unapproved/missing/reordered columns and dtype drift before reading X."""
    found = [(row[0], row[1]) for row in description]
    expected = [(name, schema["dtypes"][name]) for name in schema["columns"]]
    if found != expected:
        extra = sorted(set(name for name, _ in found) - set(schema["columns"]))
        missing = sorted(set(schema["columns"]) - set(name for name, _ in found))
        raise ValueError(f"Feature schema mismatch (columns/order/dtypes); extra={extra}, missing={missing}")


In [ ]:
"""Read-only source discovery, SQL quoting and output helpers."""
import json
from pathlib import Path


def literal(value) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def write_json(path: Path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def files_at(root: Path) -> list[Path]:
    files = sorted(root.rglob("*.parquet")) if root.is_dir() else [root] if root.is_file() else []
    if not files:
        raise FileNotFoundError(f"No Parquet files under {root}")
    return files


def parquet_scan(files) -> str:
    paths = ",".join(literal(p.as_posix()) for p in files)
    return (f"read_parquet([{paths}], hive_partitioning=false, union_by_name=false, "
            "filename=true, file_row_number=true)")


def export(con, query: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY ({query}) TO {literal(path.as_posix())} (FORMAT PARQUET, COMPRESSION ZSTD)")


def assert_empty(con, query: str, message: str):
    if con.execute(f"SELECT EXISTS ({query})").fetchone()[0]:
        raise ValueError(message)


def normalized_id(column="client_id"):
    return f"nullif(trim(cast({column} AS VARCHAR)), '')"


In [ ]:
"""Offline evaluation calendar with explicit assumed label availability."""
from datetime import date, timedelta



def next_month(day: date) -> date:
    return date(day.year + (day.month == 12), day.month % 12 + 1, 1)


def evaluation_protocol(config, months: list[str]) -> dict:
    expected = []
    day = month_start(config.first_month)
    while day <= month_start(config.last_month):
        expected.append(day.strftime("%Y-%m"))
        day = next_month(day)
    if months != expected:
        raise ValueError("Targets do not cover every configured month")
    available = {
        m: next_month(month_start(m)) + timedelta(days=config.label_delay_days)
        for m in months
    }

    def training_at(cutoff_month):
        cutoff = month_start(cutoff_month)
        result = [m for m in months if m < cutoff_month and available[m] <= cutoff]
        if not result:
            raise ValueError(f"No labels available to train at {cutoff_month}")
        return result

    calibration = config.calibration_month
    test = config.test_month
    # Do not silently retain an impossible calibration schedule when delay > 0.
    if available[calibration] > month_start(test):
        raise ValueError("Calibration labels unavailable at test cutoff; choose an earlier calibration month")
    for m in config.validation_months:
        if available[m] > month_start(calibration):
            raise ValueError("Validation labels unavailable at final base-model fit")
    return {
        "timezone": "UTC",
        "feature_rule": "event_time < cutoff; fixed-day windows are [cutoff - days, cutoff)",
        "label_availability": "Assumption, not verified publisher SLA",
        "label_delay_days": config.label_delay_days,
        "assumed_label_available_at": {m: d.isoformat() + "T00:00:00Z" for m, d in available.items()},
        "rolling_validation": [{"train": training_at(m), "validate": m} for m in config.validation_months],
        "final_train": training_at(calibration),
        "base_model_fit_at": calibration + "-01T00:00:00Z",
        "calibration": calibration,
        "test": test,
        "refit_after_calibration": False,
        "primary_budget_fraction": 0.01,
        "secondary_budget_fractions": [0.005, 0.05],
        "metrics": ["recall_at_k", "precision_at_k", "lift_at_k", "average_precision",
                    "roc_auc", "log_loss", "brier", "calibration"],
        "bootstrap_unit": "client_id",
        "note": "Existing-customer temporal evaluation; never use client_id or source fold as predictors.",
    }


In [ ]:
"""Shared data gate for CLI and both notebooks, before manifest ready."""


def validate_month(con, month, config, out, schema):
    """Validate persisted features, retaining unknown labels and missing sources."""
    file = out / f"features/month={month}/part.parquet"
    con.execute(f"CREATE OR REPLACE TEMP VIEW month_data AS SELECT * FROM "
                f"read_parquet({literal(file.as_posix())}, hive_partitioning=false)")
    validate_feature_columns(con.execute("DESCRIBE month_data").fetchall(), schema)
    assert_empty(con, "SELECT client_id, cutoff FROM month_data GROUP BY ALL HAVING count(*) <> 1",
                 "Duplicate customer-month keys")
    assert_empty(con, "SELECT 1 FROM month_data d ANTI JOIN clients c USING(client_id)",
                 "Feature ID absent from client_split")
    expected_y = ("CASE WHEN target_1=1 OR target_2=1 OR target_3=1 OR target_4=1 THEN 1 "
                  "WHEN target_1=0 AND target_2=0 AND target_3=0 AND target_4=0 THEN 0 ELSE NULL END")
    checks = {
        "Invalid month/key": f"client_id IS NULL OR cutoff IS DISTINCT FROM TIMESTAMP {literal(month + '-01')}",
        "Label eligibility mismatch": "y_any NOT IN (0,1) OR supervised_eligible IS DISTINCT FROM (y_any IS NOT NULL)",
        "Invalid source labels": " OR ".join(f"{c} NOT IN (0,1)" for c in LABELS),
        "y_any differs from source labels": f"y_any IS DISTINCT FROM ({expected_y})",
        "Non-finite/invalid predictor": " OR ".join(
            f'("{name}" IS NOT NULL AND NOT isfinite("{name}"::DOUBLE))' for name in schema["predictors"]),
        "Invalid cold_start": "cold_start IS DISTINCT FROM cast(trx_ever_seen_source=0 AND "
                              "geo_ever_seen_source=0 AND dialog_ever_seen_source=0 AS TINYINT)",
    }
    for source in SOURCES:
        for days in (*config.windows_days, None):
            prefix = f"{source}_{'all' if days is None else str(days) + 'd'}_"
            p = lambda key: prefix + key
            conditions = [
                f"{p('event_count')} < 0", f"{p('active_days')} > {p('event_count')}",
                f"{p('has_event')} IS DISTINCT FROM cast(coalesce({p('event_count')},0)>0 AS TINYINT)",
                f"{p('days_since_last_event')} < 0",
                f"{p('days_since_first_recorded_event')} < {p('days_since_last_event')}",
                f"{p('source_window_overlap_days')} < 0",
                f"({p('source_window_overlap_days')}>0 AND {p('event_count')} IS NULL)",
                f"({p('source_window_overlap_days')}=0 AND {p('event_count')} IS NOT NULL)",
            ]
            if days is not None:
                conditions.append(f"{p('days_since_first_recorded_event')} > {days}")
            if source == "trx":
                conditions += [
                    f"{p('main_count')}+{p('foreign_count')}+{p('currency_missing_count')}+{p('currency_invalid_count')} <> {p('event_count')}",
                    f"{p('known_currency_count')} <> {p('main_count')}+{p('foreign_count')}",
                    f"{p('main_amount_count')} > {p('main_count')}",
                    f"{p('has_foreign')} IS DISTINCT FROM (CASE WHEN {p('source_window_overlap_days')}<=0 THEN NULL "
                    f"WHEN coalesce({p('event_count')},0)=0 THEN 0 WHEN {p('foreign_count')}>0 THEN 1 "
                    f"WHEN {p('main_count')}>0 THEN 0 ELSE NULL END)",
                ]
            checks[prefix + "invariants"] = " OR ".join(conditions)
        tags = [str(days) + "d" for days in sorted(config.windows_days)] + ["all"]
        checks[source + " nested counts"] = " OR ".join(
            f"{source}_{a}_event_count > {source}_{b}_event_count" for a, b in zip(tags, tags[1:]))
        checks[source + " ever seen"] = f"{source}_ever_seen_source IS DISTINCT FROM {source}_all_has_event"
    checks["Invalid dialog flag"] = "dialog_has_dialog IS DISTINCT FROM dialog_ever_seen_source"
    fractions = [c for c in schema["predictors"] if "fraction" in c or "share" in c]
    checks["Invalid fraction"] = " OR ".join(f'("{c}"<0 OR "{c}">1)' for c in fractions)
    expressions = ", ".join(f"count(*) FILTER(WHERE {condition})" for condition in checks.values())
    violations = con.execute("SELECT " + expressions + " FROM month_data").fetchone()
    failed = {name: count for name, count in zip(checks, violations) if count}
    if failed:
        raise ValueError(f"Feature validation failed for {month}: {failed}")
    rows, supervised, unknown, all_missing = con.execute("""
        SELECT count(*), count(*) FILTER(WHERE supervised_eligible),
               count(*) FILTER(WHERE NOT supervised_eligible),
               count(*) FILTER(WHERE target_1 IS NULL AND target_2 IS NULL
                                  AND target_3 IS NULL AND target_4 IS NULL)
        FROM month_data""").fetchone()
    expected = con.execute("SELECT count(*) FROM clients").fetchone()[0]
    if rows != expected or rows != supervised + unknown:
        raise ValueError("Dataset counts do not reconcile")
    # Compare labels to references as well as checking their internal OR rule.
    differences = " OR ".join(f"d.{c} IS DISTINCT FROM t.{c}" for c in (*LABELS, "y_any"))
    assert_empty(con, "SELECT 1 FROM month_data d LEFT JOIN selected_targets t "
                 f"ON d.client_id=t.client_id AND d.cutoff=t.cutoff WHERE {differences}",
                 "Feature labels differ from target references")
    return dict(month=month, scoring_rows=rows, supervised_rows=supervised,
                unlabelled_rows=unknown, all_four_targets_missing=all_missing)


In [ ]:
"""Stream prepared customer-month features into one verified local Parquet."""
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path



def merge_prepared(source_dir, output_dir, batch_size=10_000):
    """Concatenate already joined months; never join raw one-to-many event logs."""
    import duckdb
    import pyarrow as pa
    import pyarrow.parquet as pq

    if type(batch_size) is not int or batch_size < 1:
        raise ValueError("batch_size must be a positive integer")
    source, out = Path(source_dir).resolve(), Path(output_dir).resolve()
    read = lambda path: json.loads(path.read_text(encoding="utf-8"))
    metadata_paths = [source / name for name in
                      ("manifest.json", "feature_schema.json", "protocol.json", "work.duckdb")]
    prepared = read(metadata_paths[0])
    if prepared.get("status") != "ready":
        raise ValueError("Source manifest must be ready")
    config = PreparationConfig(**prepared["config"])
    for protected in (source, config.data_root):
        if out == protected or out in protected.parents or protected in out.parents:
            raise ValueError("Merged output must not overlap prepared or raw inputs")
    schema = feature_schema(config)
    if read(metadata_paths[1]) != schema:
        raise ValueError("Source schema must match the approved schema v2")
    months = [row["month"] for row in prepared["months"]]
    protocol = evaluation_protocol(config, months)
    if read(metadata_paths[2]) != protocol:
        raise ValueError("Source protocol differs from the evaluation calendar")
    paths = [source / f"features/month={month}/part.parquet" for month in months]

    def inventory():
        return [{"path": p.relative_to(source).as_posix(), "bytes": p.stat().st_size,
                 "mtime_ns": p.stat().st_mtime_ns} for p in (*metadata_paths, *paths)]

    before = inventory()  # Fail on absent inputs before creating output.
    out.mkdir(parents=True, exist_ok=False)
    partial = out / "propensity.incomplete.parquet"
    destination = out / "propensity.parquet"
    manifest = {"status": "running", "started_at": datetime.now(timezone.utc).isoformat(),
                "source_dir": str(source), "source_inventory_sha256": prepared["source_inventory_sha256"],
                "schema_version": schema["schema_version"], "operation": "concatenate_customer_month_partitions",
                "input_files": before, "months": [], "batch_size": batch_size,
                "raw_modified": False, "models_fitted": False}
    write_json(out / "manifest.json", manifest)
    try:
        with duckdb.connect(str(source / "work.duckdb"), read_only=True) as con:
            con.execute("SET memory_limit='2GB'")
            con.execute("SET threads=2")
            con.execute(f"SET temp_directory={literal((out / 'spill').as_posix())}")
            writer = None
            arrow_types = {"VARCHAR": pa.string(), "TIMESTAMP": pa.timestamp("us"),
                           "TINYINT": pa.int8(), "BIGINT": pa.int64(),
                           "DOUBLE": pa.float64(), "BOOLEAN": pa.bool_()}
            arrow_schema = pa.schema([(name, arrow_types[schema["dtypes"][name]])
                                      for name in schema["columns"]])
            try:
                for summary, path in zip(prepared["months"], paths):
                    month = summary["month"]
                    check = validate_month(con, month, config, source, schema)
                    positive = con.execute("SELECT coalesce(sum(y_any),0) FROM month_data").fetchone()[0]
                    if (check["scoring_rows"] != summary["rows"] or
                            check["supervised_rows"] != summary["labelled_rows"] or
                            check["unlabelled_rows"] != summary["unknown_labels"] or
                            positive != summary["positives"]):
                        raise ValueError(f"Source manifest counts differ for {month}")
                    parquet = pq.ParquetFile(path)
                    if writer is None:
                        writer = pq.ParquetWriter(partial, arrow_schema, compression="zstd")
                    written = 0
                    for batch in parquet.iter_batches(batch_size=batch_size):
                        # Normalize equivalent Arrow encodings/nullability only after
                        # SQL names/types passed the fixed contract; casts are safe.
                        writer.write_batch(batch.cast(arrow_schema, safe=True))
                        written += batch.num_rows
                    if written != summary["rows"]:
                        raise ValueError(f"Incomplete streamed month {month}")
                    manifest["months"].append(dict(summary))
                    write_json(out / "manifest.json", manifest)
                    print(f"Merged {month}: {written:,} rows", flush=True)
            finally:
                if writer is not None:
                    writer.close()
            con.execute(f"CREATE OR REPLACE TEMP VIEW merged_data AS SELECT * FROM "
                        f"read_parquet({literal(partial.as_posix())}, hive_partitioning=false)")
            validate_feature_columns(con.execute("DESCRIBE merged_data").fetchall(), schema)
            rows, unique, invalid_keys = con.execute("""SELECT count(*),
                count(DISTINCT (client_id,cutoff)),
                count(*) FILTER(WHERE client_id IS NULL OR cutoff IS NULL) FROM merged_data""").fetchone()
            actual = con.execute("""SELECT strftime(cutoff,'%Y-%m'), count(*), count(y_any),
                coalesce(sum(y_any),0), count(*) FILTER(WHERE y_any IS NULL), sum(cold_start)
                FROM merged_data GROUP BY cutoff ORDER BY cutoff""").fetchall()
            expected = [(r["month"], r["rows"], r["labelled_rows"], r["positives"],
                         r["unknown_labels"], r["cold_start_rows"]) for r in prepared["months"]]
            if rows != unique or invalid_keys or rows != sum(r["rows"] for r in prepared["months"]) or actual != expected:
                raise ValueError("Merged keys/monthly counts do not reconcile")
        if inventory() != before:
            raise ValueError("Prepared inputs changed during merge")
        write_json(out / "feature_schema.json", schema)
        write_json(out / "protocol.json", protocol)
        digest = hashlib.sha256()
        with partial.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        partial.rename(destination)
        manifest.update(status="ready", rows=rows, columns=len(schema["columns"]),
                        predictors=len(schema["predictors"]), unique_customer_months=unique,
                        positives=sum(r["positives"] for r in prepared["months"]),
                        unknown_labels=sum(r["unknown_labels"] for r in prepared["months"]),
                        dataset="propensity.parquet", bytes=destination.stat().st_size,
                        dataset_sha256=digest.hexdigest(), finished_at=datetime.now(timezone.utc).isoformat())
    except BaseException as exc:
        manifest.update(status="failed", error=f"{type(exc).__name__}: {exc}")
        raise
    finally:
        write_json(out / "manifest.json", manifest)
    return out / "manifest.json"


## 4. Kiểm kê trước merge

Chỉ đọc manifest ở bước này. Bước 5 sẽ kiểm tra schema, giá trị, khóa, nhãn và
quần thể bằng validation dùng chung trước khi ghi từng tháng.


In [ ]:
source_manifest = json.loads((SOURCE_DIR / "manifest.json").read_text(encoding="utf-8"))
if source_manifest.get("status") != "ready":
    raise ValueError("Nguồn chưa ready; hoàn tất notebook 03 trước.")
display(pd.DataFrame(source_manifest["months"]))
print("Expected rows:", sum(row["rows"] for row in source_manifest["months"]))


## 5. Merge và lưu

Ghi Parquet nén ZSTD theo batch, không nạp toàn bộ ma trận vào pandas. Đối soát
khóa duy nhất, số dòng/nhãn/cold-start từng tháng và inventory đầu-cuối.
Chỉ đổi tên file tạm thành `propensity.parquet` khi kiểm tra đạt. Output lỗi có
manifest `failed`; giữ lại để kiểm tra, dùng thư mục mới khi chạy lại.


In [ ]:
merged_manifest_path = merge_prepared(SOURCE_DIR, OUTPUT_DIR, batch_size=BATCH_SIZE)
print(merged_manifest_path)


## 6. Kiểm tra kết quả

`propensity.parquet` giữ khóa, 221 predictor mặc định, bốn nhãn gốc, `y_any` và
`supervised_eligible` (229 cột mặc định). Chỉ chọn `schema["predictors"]` làm X;
chỉ train/evaluate dòng có `supervised_eligible=true`. Giữ dòng thiếu nhãn để scoring.
Dùng split theo tháng trong protocol, không random-split khách–tháng.


In [ ]:
merged_manifest = json.loads(merged_manifest_path.read_text(encoding="utf-8"))
if merged_manifest.get("status") != "ready":
    raise RuntimeError("Chỉ đọc bộ merge có manifest ready.")
merged_schema = json.loads((OUTPUT_DIR / "feature_schema.json").read_text(encoding="utf-8"))
merged_protocol = json.loads((OUTPUT_DIR / "protocol.json").read_text(encoding="utf-8"))
parquet = pq.ParquetFile(OUTPUT_DIR / merged_manifest["dataset"])
assert parquet.metadata.num_rows == merged_manifest["rows"]
assert parquet.schema_arrow.names == merged_schema["columns"]
display(pd.Series({k: merged_manifest[k] for k in
                   ("status", "rows", "columns", "predictors", "positives", "unknown_labels", "bytes")}))
# Khi đọc mô hình, dùng iter_batches với columns=[*merged_schema["keys"],
# *merged_schema["predictors"], "y_any", "supervised_eligible"], rồi lọc eligibility.


Các file sau merge:

```text
processed/
├── propensity.parquet
├── feature_schema.json
├── protocol.json
└── manifest.json
```

Không đưa dữ liệu sau merge lên Git. Notebook nguồn giữ `execution_count=null`
và output trống. Không có kết quả merge thật hoặc huấn luyện trong bản notebook này.
